# **Metrics Extractor**

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Clean.csv')

Full function

In [3]:
def extract_metrics(df, target_column=None):
    df = df.copy()
    
    # Normalize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.replace("\n", " ", regex=False)
    )
    
    metrics = {}
    
    # Dataset-level metrics
    metrics["dataset"] = {
        "n_rows": len(df),
        "n_columns": df.shape[1],
        "is_empty": df.empty
    }
    
    # Schema & column roles
    dtypes = df.dtypes.astype(str).to_dict()
    
    metrics["schema"] = {
        "columns": df.columns.tolist(),
        "dtypes": dtypes,
        "column_roles": {
            "numeric": df.select_dtypes(include="number").columns.tolist(),
            "categorical": df.select_dtypes(include="object").columns.tolist(),
            "boolean": df.select_dtypes(include="bool").columns.tolist()
        }
    }
    
    # Missingness metrics
    missing_per_column = df.isna().mean().to_dict()
    missing_per_row = df.isna().mean(axis=1).to_list()
    
    metrics["missingness"] = {
        "per_column_missing_pct": missing_per_column,
        "per_row_missing_pct": missing_per_row,
        "any_missing": any(v > 0 for v in missing_per_column.values())
    }
    
    # Uniqueness & cardinality
    nunique_per_column = df.nunique(dropna=False).to_dict()
    
    uniqueness_ratio = {
        col: nunique_per_column[col] / len(df) if len(df) > 0 else 0
        for col in df.columns
    }
    
    constant_columns = [col for col, n in nunique_per_column.items() if n == 1]
    
    identifier_like_columns = [
        col for col, r in uniqueness_ratio.items()
        if r > 0.95
    ]
    
    metrics["uniqueness"] = {
        "nunique_per_column": nunique_per_column,
        "uniqueness_ratio_per_column": uniqueness_ratio,
        "constant_columns": constant_columns,
        "identifier_like_columns": identifier_like_columns
    }
    
    # Target metrics
    metrics["target"] = {
        "name": target_column,
        "exists": target_column in df.columns if target_column else False
    }
    
    if target_column and target_column in df.columns:
        target_series = df[target_column]
        
        target_dtype = str(target_series.dtype)
        
        if target_series.dtype == bool:
            target_type = "boolean"
        elif target_series.dtype == object:
            target_type = "categorical"
        else:
            target_type = "numeric"
        
        metrics["target"].update({
            "dtype": target_dtype,
            "type": target_type,
            "missing_pct": target_series.isna().mean(),
            "nunique": target_series.nunique(dropna=False),
            "uniqueness_ratio": (
                target_series.nunique(dropna=False) / len(df) if len(df) > 0 else 0
            )
        })
        
        if target_type == "numeric":
            metrics["target"]["distribution_stats"] = target_series.describe().to_dict()
        
        if target_type in ["boolean", "categorical"]:
            vc = target_series.value_counts(normalize=True, dropna=False).to_dict()
            metrics["target"]["value_counts"] = vc
            
            if target_type == "boolean":
                metrics["target"]["balance_rate"] = target_series.mean()
    
    # Numeric distribution metrics
    numeric_cols = df.select_dtypes(include="number").columns
    
    dist_stats = {}
    
    for col in numeric_cols:
        desc = df[col].describe()
        dist_stats[col] = {
            "min": desc["min"],
            "max": desc["max"],
            "mean": desc["mean"],
            "std": desc["std"],
            "25%": desc["25%"],
            "50%": desc["50%"],
            "75%": desc["75%"]
        }
    
    metrics["distribution"] = {
        "per_numeric_column_stats": dist_stats
    }
    
    # Formatting & type-confusion flags
    currency_like = []
    comma_numeric = []
    mixed_type = []
    
    for col in df.select_dtypes(include="object").columns:
        series = df[col].dropna().astype(str)
        
        if series.str.contains("$", regex=False).any():
            currency_like.append(col)
        
        if series.str.contains(",", regex=False).any():
            comma_numeric.append(col)
        
        numeric_ratio = series.str.replace(".", "", regex=False).str.isnumeric().mean()
        if 0 < numeric_ratio < 1:
            mixed_type.append(col)
    
    metrics["formatting"] = {
        "currency_like_columns": currency_like,
        "comma_separated_numeric_columns": comma_numeric,
        "mixed_type_columns": mixed_type
    }
    
    return metrics


Test Cell

In [4]:
metrics = extract_metrics(df, target_column="average_score")
metrics.keys()


dict_keys(['dataset', 'schema', 'missingness', 'uniqueness', 'target', 'distribution', 'formatting'])

# **Target-Aware Router**

Router Interface and Target Validation

In [5]:
def route_checks(metrics, dataset_type):
    target_info = metrics.get("target", {})
    
    # Hard stop if no target
    if not target_info.get("exists", False):
        return {
            "status": "stop",
            "reason": "Target column does not exist",
            "task_type": None,
            "dataset_type": dataset_type,
            "run_interpreters": [],
            "skip_modules": ["all"]
        }
    
    # Determine task type
    target_type = target_info.get("type")
    
    if target_type == "numeric":
        task_type = "regression"
    elif target_type == "boolean":
        task_type = "binary_classification"
    elif target_type == "categorical":
        task_type = "multiclass_classification"
    else:
        task_type = "unknown"
    
    run_interpreters = ["core"]
    skip_modules = []
    
    # Dataset-type routing
    if dataset_type == "clean":
        run_interpreters.append("clean")
        skip_modules.append("messy")
    elif dataset_type == "messy":
        run_interpreters.append("messy")
        skip_modules.append("clean")
    elif dataset_type == "biased":
        pass
    
    # Task-type routing
    if task_type == "regression":
        run_interpreters.append("regression")
        skip_modules.append("bias")
    
    elif task_type == "binary_classification":
        run_interpreters.append("bias")
        skip_modules.append("regression")
    
    elif task_type == "multiclass_classification":
        run_interpreters.append("classification")
        skip_modules.append("regression")
    
    router_config = {
        "status": "ok",
        "task_type": task_type,
        "dataset_type": dataset_type,
        "run_interpreters": run_interpreters,
        "skip_modules": skip_modules
    }
    
    return router_config


Test Cell

In [6]:
router_config = route_checks(metrics, dataset_type="clean")
router_config


{'status': 'ok',
 'task_type': 'regression',
 'dataset_type': 'clean',
 'run_interpreters': ['core', 'clean', 'regression'],
 'skip_modules': ['messy', 'bias']}

# **Core Interpreter**

Full function

In [7]:
def core_interpreter(metrics):
    findings = {}
    
    # Missingness findings
    missing_info = metrics.get("missingness", {})
    per_col_missing = missing_info.get("per_column_missing_pct", {})
    
    missing_columns = [col for col, pct in per_col_missing.items() if pct > 0]
    high_missing_columns = {
        col: pct for col, pct in per_col_missing.items() if pct > 0.3
    }
    
    findings["missing_columns"] = missing_columns
    findings["high_missing_columns"] = high_missing_columns
    
    # Uniqueness findings
    uniq_info = metrics.get("uniqueness", {})
    
    constant_columns = uniq_info.get("constant_columns", [])
    identifier_like_columns = uniq_info.get("identifier_like_columns", [])
    
    findings["constant_columns"] = constant_columns
    findings["identifier_like_columns"] = identifier_like_columns
    
    # Target validity findings
    target_info = metrics.get("target", {})
    
    target_findings = {}
    
    if not target_info.get("exists", False):
        target_findings["exists"] = False
    else:
        target_findings["exists"] = True
        target_findings["missing_pct"] = target_info.get("missing_pct")
        target_findings["type"] = target_info.get("type")
        target_findings["uniqueness_ratio"] = target_info.get("uniqueness_ratio")
    
    findings["target"] = target_findings
    
    # Leakage candidates
    leakage_candidates = {}
    
    leakage_candidates["identifier_like_columns"] = (
        metrics.get("uniqueness", {}).get("identifier_like_columns", [])
    )
    
    findings["leakage_candidates"] = leakage_candidates
    
    return findings


Test Cell

In [8]:
core_findings = core_interpreter(metrics)
core_findings


{'missing_columns': [],
 'high_missing_columns': {},
 'constant_columns': [],
 'identifier_like_columns': [],
 'target': {'exists': True,
  'missing_pct': np.float64(0.0),
  'type': 'numeric',
  'uniqueness_ratio': 0.194},
 'leakage_candidates': {'identifier_like_columns': []}}

 **Clean Interpreter**

Function

In [9]:
def clean_interpreter(metrics):
    findings = {}
    
    # Missingness in clean data should be near zero
    missing_info = metrics.get("missingness", {})
    per_col_missing = missing_info.get("per_column_missing_pct", {})
    
    any_missing = [col for col, pct in per_col_missing.items() if pct > 0]
    
    findings["any_missing_columns"] = any_missing
    
    # Type stability issues (object columns that look numeric)
    formatting_info = metrics.get("formatting", {})
    
    mixed_type_columns = formatting_info.get("mixed_type_columns", [])
    currency_like_columns = formatting_info.get("currency_like_columns", [])
    comma_numeric_columns = formatting_info.get("comma_separated_numeric_columns", [])
    
    findings["mixed_type_columns"] = mixed_type_columns
    findings["currency_like_columns"] = currency_like_columns
    findings["comma_numeric_columns"] = comma_numeric_columns
    
    # Numeric sanity checks
    dist_info = metrics.get("distribution", {}).get("per_numeric_column_stats", {})
    
    zero_variance_columns = []
    
    for col, stats in dist_info.items():
        if stats["std"] == 0:
            zero_variance_columns.append(col)
    
    findings["zero_variance_numeric_columns"] = zero_variance_columns
    
    return findings


Test Cell

In [ ]:
clean_findings = clean_interpreter(metrics)
clean_findings

{'any_missing_columns': [],
 'mixed_type_columns': [],
 'currency_like_columns': [],
 'comma_numeric_columns': [],
 'zero_variance_numeric_columns': []}

**Messy Interpreter**

Function

In [13]:
def messy_interpreter(metrics):
    findings = {}
    
    # Missingness profile (messy tolerates missing, but records distribution)
    missing_info = metrics.get("missingness", {})
    per_col_missing = missing_info.get("per_column_missing_pct", {})
    
    high_missing_columns = {
        col: pct for col, pct in per_col_missing.items() if pct > 0.3
    }
    
    moderate_missing_columns = {
        col: pct for col, pct in per_col_missing.items() if 0 < pct <= 0.3
    }
    
    findings["high_missing_columns"] = high_missing_columns
    findings["moderate_missing_columns"] = moderate_missing_columns
    
    # Formatting issues (core messy signals)
    formatting_info = metrics.get("formatting", {})
    
    mixed_type_columns = formatting_info.get("mixed_type_columns", [])
    currency_like_columns = formatting_info.get("currency_like_columns", [])
    comma_numeric_columns = formatting_info.get("comma_separated_numeric_columns", [])
    
    findings["mixed_type_columns"] = mixed_type_columns
    findings["currency_like_columns"] = currency_like_columns
    findings["comma_numeric_columns"] = comma_numeric_columns
    
    # Cardinality extremes (too low or too high)
    uniq_info = metrics.get("uniqueness", {})
    uniqueness_ratio = uniq_info.get("uniqueness_ratio_per_column", {})
    
    low_cardinality_columns = [
        col for col, r in uniqueness_ratio.items() if r < 0.01
    ]
    
    high_cardinality_columns = [
        col for col, r in uniqueness_ratio.items() if r > 0.9
    ]
    
    findings["low_cardinality_columns"] = low_cardinality_columns
    findings["high_cardinality_columns"] = high_cardinality_columns
    
    return findings


Test Cell *(No values because tested dataset is Clean)*

In [14]:
messy_findings = messy_interpreter(metrics)
messy_findings


{'high_missing_columns': {},
 'moderate_missing_columns': {},
 'mixed_type_columns': [],
 'currency_like_columns': [],
 'comma_numeric_columns': [],
 'low_cardinality_columns': ['gender',
  'race_ethnicity',
  'parental_level_of_education',
  'lunch',
  'test_preparation_course'],
 'high_cardinality_columns': []}

**Regression Interpreter**

Function

In [15]:
def regression_interpreter(metrics):
    findings = {}
    
    # Target distribution analysis
    target_info = metrics.get("target", {})
    
    target_findings = {}
    
    if target_info.get("type") == "numeric":
        dist = target_info.get("distribution_stats", {})
        
        target_findings["min"] = dist.get("min")
        target_findings["max"] = dist.get("max")
        target_findings["mean"] = dist.get("mean")
        target_findings["std"] = dist.get("std")
        
        # Simple skew proxy: mean vs median
        median = dist.get("50%")
        mean = dist.get("mean")
        
        if median is not None and mean is not None:
            target_findings["mean_minus_median"] = mean - median
        
        # Range sanity
        if dist.get("min") is not None and dist.get("max") is not None:
            target_findings["range"] = dist.get("max") - dist.get("min")
    
    findings["target_distribution"] = target_findings
    
    # Numeric feature sanity
    dist_info = metrics.get("distribution", {}).get("per_numeric_column_stats", {})
    
    zero_variance_features = []
    extreme_range_features = []
    
    for col, stats in dist_info.items():
        if stats["std"] == 0:
            zero_variance_features.append(col)
        
        if stats["min"] is not None and stats["max"] is not None:
            if stats["max"] > 1e6 * max(1, abs(stats["min"])):
                extreme_range_features.append(col)
    
    findings["zero_variance_features"] = zero_variance_features
    findings["extreme_range_features"] = extreme_range_features
    
    return findings


Test Cell

In [16]:
regression_findings = regression_interpreter(metrics)
regression_findings


{'target_distribution': {'min': 9.0,
  'max': 100.0,
  'mean': 67.77066666666666,
  'std': 14.257325984669146,
  'mean_minus_median': -0.5626666666666722,
  'range': 91.0},
 'zero_variance_features': [],
 'extreme_range_features': []}

**Bias Interpreter**

Function

In [19]:
def bias_interpreter(metrics):
    findings = {}
    
    target_info = metrics.get("target", {})
    
    # Bias interpreter only valid for boolean targets
    if target_info.get("type") != "boolean":
        findings["status"] = "skipped"
        findings["reason"] = "Target is not boolean"
        return findings
    
    # Baseline rate
    baseline_rate = target_info.get("balance_rate")
    
    findings["baseline_rate"] = baseline_rate
    
    # Group-level outcome rates
    group_bias = {}
    
    schema_info = metrics.get("schema", {})
    candidate_group_columns = (
        schema_info.get("column_roles", {}).get("categorical", []) +
        schema_info.get("column_roles", {}).get("boolean", [])
    )
    
    findings["candidate_sensitive_columns"] = candidate_group_columns
    findings["group_outcome_rates"] = {}  # placeholder for later extension
    
    return findings


Test Cell

In [ ]:
bias_findings = bias_interpreter(metrics)
bias_findings

{'status': 'skipped', 'reason': 'Target is not boolean'}

## Analysis Layer

Implemented the complete **analysis engine**

Components built:

- Unified Metrics Extractor  
- Target-Aware Router  
- Core Interpreter  
- Clean Interpreter  
- Messy Interpreter  
- Regression Interpreter  
- Bias Interpreter  




